In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("./s1_data_hw.csv", encoding="utf-8-sig")
df

,Date,SXXP Index,SPY US INDEX,NKY INDEX,XAU CURNCY,CL1 COMDTY,SPDIV,RF,DFEDTARU
0,12/31/08,276.76878,90.24,98.07993,875.43,44.60,28.390000,2.87,0.25
1,1/30/09,245.27160,82.83,88.89192,927.85,41.68,28.013333,3.02,0.25
2,2/27/09,219.88507,73.93,77.26031,942.32,44.76,27.636667,2.71,0.25
3,3/31/09,233.43893,79.52,81.62587,919.90,49.66,27.260000,3.16,0.25
4,4/30/09,264.48381,87.42,89.27354,887.95,51.12,26.703333,3.47,0.25
...,...,...,...,...,...,...,...,...,...
200,8/29/25,643.77383,645.05,290.77987,3447.95,64.01,78.103333,4.16,4.25
201,9/30/25,656.25223,666.18,304.13314,3846.09,62.37,78.480000,4.11,4.00
202,10/31/25,659.04604,682.06,340.11252,4002.92,60.98,78.626720,4.02,4.00
203,11/28/25,668.83173,683.39,321.70738,4239.43,58.55,78.773440,4.18,3.75


In [3]:
from homework01 import monthly_log_return

In [4]:
monthly_returns, df_adjusted = monthly_log_return(df)

Mean monthly log return STOXX600: 0.0063
Mean monthly log return NIKKEI: 0.0034


In [5]:
monthly_returns.head()

,Date,RF_monthly,SP500_monthly_log_return,STOXX600_monthly_log_return,NIKKEI_monthly_log_return,GOLD_monthly_log_return,WTI_monthly_log_return
0,2021-02-26,0.001450,0.0274,0.0185,0.0276,-0.0635,0.1640
1,2021-03-31,0.001375,0.0411,0.0313,-0.0291,-0.0128,-0.0388
2,2021-04-30,0.001317,0.0516,0.0413,-0.0007,0.0328,0.0721
3,2021-05-31,0.001208,0.0065,0.0378,0.0000,0.0749,0.0422
4,2021-06-30,0.001033,0.0189,-0.0180,-0.0170,-0.0731,0.1024


In [6]:
df_adjusted.head()

,Date,SXXP Index,SPY US INDEX,NKY INDEX,XAU CURNCY,CL1 COMDTY,SPDIV,RF,DFEDTARU,RF_monthly,SP500_monthly_log_return,STOXX600_monthly_log_return,NIKKEI_monthly_log_return,GOLD_monthly_log_return,WTI_monthly_log_return
145,2021-01-29,480.24522,370.07,264.21576,1847.65,52.20,58.063693,1.44,0.25,0.001200,NaN,NaN,NaN,NaN,NaN
146,2021-02-26,489.22792,380.36,271.59878,1734.04,61.50,57.848540,1.74,0.25,0.001450,0.0274,0.0185,0.0276,-0.0635,0.1640
147,2021-03-31,504.78000,396.33,263.79893,1712.02,59.16,57.633387,1.65,0.25,0.001375,0.0411,0.0313,-0.0291,-0.0128,-0.0388
148,2021-04-30,526.04895,417.30,263.61052,1769.13,63.58,57.710605,1.58,0.25,0.001317,0.0516,0.0413,-0.0007,0.0328,0.0721
149,2021-05-31,546.29813,420.04,263.61052,1906.75,66.32,57.787824,1.45,0.25,0.001208,0.0065,0.0378,0.0000,0.0749,0.0422


In [7]:
def monthly_log_return(df):
    """Return monthly log returns from January 2021 onward."""
    data = df.copy()
    data = data.rename(columns=lambda col: col.strip().replace("\ufeff", ""))
    data["Date"] = pd.to_datetime(data["Date"], format="%m/%d/%y")
    data = data.sort_values("Date")
    start_date = pd.Timestamp("2021-01-01")
    data = data[data["Date"] >= start_date].copy()
    data["RF_monthly"] = (data["RF"] / 100) / 12

    assets = {
        "SP500": "SPY US INDEX",
        "STOXX600": "SXXP Index",
        "NIKKEI": "NKY INDEX",
        "GOLD": "XAU CURNCY",
        "WTI": "CL1 COMDTY",
    }

    for label, column in assets.items():
        log_returns = np.log(data[column]).diff().round(4)
        data[f"{label}_monthly_log_return"] = log_returns

    result_cols = ["Date", "RF_monthly"] + [f"{label}_monthly_log_return" for label in assets]
    result = data[result_cols].dropna().reset_index(drop=True)

    sto_mean = result["STOXX600_monthly_log_return"].mean()
    nky_mean = result["NIKKEI_monthly_log_return"].mean()
    print(f"Mean monthly log return STOXX600: {sto_mean:.4f}")
    print(f"Mean monthly log return NIKKEI: {nky_mean:.4f}")

    return result, data


def wti_plots(df):
    price_series = df[["Date", "CL1 COMDTY"]].dropna()
    return_series = df.dropna(subset=["WTI_monthly_log_return"])

    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    axes[0].plot(price_series["Date"], price_series["CL1 COMDTY"], color="tab:blue")
    axes[0].axhline(price_series["CL1 COMDTY"].mean(), color="black", linewidth=0.8, linestyle="--")
    axes[0].set_title("WTI Price Index")
    axes[0].set_ylabel("Index Level")

    axes[1].plot(return_series["Date"], return_series["WTI_monthly_log_return"], color="tab:orange")
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title("WTI Monthly Log Returns")
    axes[1].set_ylabel("Log Return")
    axes[1].set_xlabel("Date")

    plt.tight_layout()
    plt.show()

    max_idx = return_series["WTI_monthly_log_return"].abs().idxmax()
    max_date = return_series.loc[max_idx, "Date"].strftime("%b %Y")
    max_value = return_series.loc[max_idx, "WTI_monthly_log_return"]
    print(
        "WTI returns show a pronounced spike of "
        f"{max_value:.2%} around {max_date}, highlighting their volatility.\n"
    )


def testing_avg_ret_stoxx(df): 
    # HYPOTHESIS:
    # H_0:= mu = 0 
    # H_1:= mu != 0

    significance_level = 0.05
    t, p = stats.ttest_1samp(df["STOXX600_monthly_log_return"], popmean=0.0)

    print(f"t-statistic: ${t}\n")
    print(f"p-value: ${p}\n")
    if p < significance_level:
        print("REJECT H_0\n")
    else: 
        print("FAILED TO REJECT H_0\n")
    
    # *spoiler*: we fail to reject the H_0 => H_0 is likely


def testing_CI_avg_ret_gold(df): 
    level = 0.95
    gold_returns = df["GOLD_monthly_log_return"].dropna()
    n = gold_returns.count()
    mean_value = gold_returns.mean()
    se = gold_returns.std(ddof=1) / np.sqrt(n)
    a, b = stats.t.interval(level, n - 1, loc=mean_value, scale=se)
    
    print(f"CI-Testing: 2.5th percentile: ${a}\n")
    print(f"CI-Testing: 97.5th percentile: ${b}\n")
    rf_mean = df["RF_monthly"].mean()
    print(f"Average monthly risk-free rate: {rf_mean:.4f}")
    if a > rf_mean:
        print("Gold's mean return exceeds the risk-free rate at the 95% confidence level.\n")
    else:
        print("Cannot claim Gold outperformed the risk-free rate at the 95% confidence level.\n")

def correlation_matrix_risky_assets(monthly_returns): 
    risky_cols = [
        "SP500_monthly_log_return",
        "STOXX600_monthly_log_return",
        "NIKKEI_monthly_log_return",
        "GOLD_monthly_log_return",
        "WTI_monthly_log_return",
    ]
    
    correlation_matrix = monthly_returns[risky_cols].corr()
    return correlation_matrix

In [ ]:
correlation_matrix = correlation_matrix_risky_assets(monthly_returns)
wti_max_corr = (
    correlation_matrix["WTI_monthly_log_return"]
    .drop("WTI_monthly_log_return")
    .idxmax()
)

In [10]:
correlation_matrix

,SP500_monthly_log_return,STOXX600_monthly_log_return,NIKKEI_monthly_log_return,GOLD_monthly_log_return,WTI_monthly_log_return
SP500_monthly_log_return,1.000000,0.772483,0.690575,0.140583,0.099431
STOXX600_monthly_log_return,0.772483,1.000000,0.632987,0.394984,0.068535
NIKKEI_monthly_log_return,0.690575,0.632987,1.000000,0.229523,-0.048498
GOLD_monthly_log_return,0.140583,0.394984,0.229523,1.000000,-0.217613
WTI_monthly_log_return,0.099431,0.068535,-0.048498,-0.217613,1.000000


In [14]:
def highest_kurtoisis(monthly_returns): 
    risky_cols = [
        "SP500_monthly_log_return",
        "STOXX600_monthly_log_return",
        "NIKKEI_monthly_log_return",
        "GOLD_monthly_log_return",
        "WTI_monthly_log_return",
    ]

    return monthly_returns[risky_cols].kurtosis()    

In [17]:
monthly_returns_kurtosis = highest_kurtoisis(monthly_returns)
monthly_returns_kurtosis

SP500_monthly_log_return      -0.012166
STOXX600_monthly_log_return   -0.130535
NIKKEI_monthly_log_return     -0.002500
GOLD_monthly_log_return       -0.488413
WTI_monthly_log_return         0.343564
dtype: float64